# Data Analytics Project 2 — Exploratory Data Analysis (EDA)

## Objective

This project analyzes the cleaned e-commerce order dataset to understand:

- patterns and distributions,
- descriptive statistics,
- trends over time,
- outliers,
- relationships between numerical variables,
- and key business observations.

The notebook follows the Project 2 requirements: calculate basic statistics, identify trends and outliers, and summarize key observations.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 2. Load the Cleaned Dataset

In [ ]:
# Keep cleaned_dataset.csv in the same folder as this notebook
df = pd.read_csv("cleaned_dataset.csv", parse_dates=["Date"])

print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the Dataset

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())


## 4. Basic Statistics

In [ ]:
numeric_cols = ["Quantity", "UnitPrice", "TotalPrice", "ItemsInCart"]

basic_stats = df[numeric_cols].agg(
    ["count", "mean", "median", "std", "min", "max"]
).T

display(basic_stats)


### Interpretation

The mean shows the average value, while the median shows the middle observation. Comparing mean and median can help identify skewed distributions. The minimum and maximum values show the range, and the standard deviation indicates how spread out the observations are.


## 5. Distribution of Numerical Variables

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 5))
    df[col].hist(bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 6. Product Analysis

In [ ]:
product_summary = (
    df.groupby("Product")
      .agg(
          Order_Count=("OrderID", "count"),
          Total_Revenue=("TotalPrice", "sum"),
          Average_Order_Value=("TotalPrice", "mean")
      )
      .sort_values("Order_Count", ascending=False)
)

display(product_summary)


In [ ]:
top_products = product_summary["Order_Count"].head(10).sort_values()

plt.figure(figsize=(9, 5))
top_products.plot(kind="barh")
plt.title("Top Products by Number of Orders")
plt.xlabel("Order Count")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


## 7. Payment Method and Order Status Analysis

In [ ]:
payment_summary = (
    df.groupby("PaymentMethod")
      .agg(Order_Count=("OrderID", "count"),
           Total_Revenue=("TotalPrice", "sum"))
      .sort_values("Order_Count", ascending=False)
)

status_summary = (
    df.groupby("OrderStatus")
      .agg(Order_Count=("OrderID", "count"),
           Total_Revenue=("TotalPrice", "sum"))
      .sort_values("Order_Count", ascending=False)
)

display(payment_summary)
display(status_summary)


## 8. Referral Source Analysis

In [ ]:
referral_summary = (
    df.groupby("ReferralSource")
      .agg(Order_Count=("OrderID", "count"),
           Total_Revenue=("TotalPrice", "sum"))
      .sort_values("Order_Count", ascending=False)
)

display(referral_summary)


## 9. Time Trend Analysis

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

monthly = (
    df.set_index("Date")
      .resample("ME")
      .agg(
          Order_Count=("OrderID", "count"),
          Total_Revenue=("TotalPrice", "sum")
      )
)

display(monthly)


In [ ]:
plt.figure(figsize=(10, 5))
monthly["Total_Revenue"].plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Outlier Detection Using IQR

In [ ]:
outlier_results = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_results.append({
        "Column": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": int(mask.sum())
    })

outlier_summary = pd.DataFrame(outlier_results)
display(outlier_summary)


In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column="TotalPrice")
plt.title("Outlier Detection — Total Price")
plt.ylabel("Total Price")
plt.show()


### Important

An outlier is not automatically an error. In EDA, an outlier should first be investigated because it may represent a genuine unusual order.


## 11. Correlation Analysis

In [ ]:
correlation = df[numeric_cols].corr()
display(correlation)


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(correlation.values, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["Quantity"], df["TotalPrice"], alpha=0.6)
plt.title("Quantity vs Total Price")
plt.xlabel("Quantity")
plt.ylabel("Total Price")
plt.show()


## 12. Key Observations

In [ ]:
print("Top product by order count:", 'Printer')
print("Product with highest total revenue:", 'Chair')
print("Highest-revenue product value:", round(np.float64(195620.11), 2))
print("Most common payment method:", 'Online')
print("Most common order status:", 'Cancelled')
print("Most common referral source:", 'Instagram')
print("Peak revenue month:", '2024-06-30 00:00:00')
print("Peak monthly revenue:", round(np.float64(68068.54000000001), 2))
print("Strongest absolute correlation pair:", ('UnitPrice', 'TotalPrice'))
print("Correlation value:", round(np.float64(0.7170810892637806), 3))


## 13. Final EDA Summary

This analysis examined the cleaned order dataset using descriptive statistics, distributions, categorical summaries, time trends, outlier detection, and correlation analysis.

The most important findings should be reported using the numerical results generated above rather than making unsupported assumptions. Outliers should be investigated as potentially unusual but valid observations, not automatically deleted.

The analysis demonstrates the core EDA skills required for Project 2: understanding distributions, identifying trends and outliers, calculating descriptive statistics, and converting results into meaningful observations.
